
# 03 — Consolidação dos Resultados — UTKFace

Este notebook tem como objetivo consolidar os resultados obtidos pelas tecnologias DeepFace e MiVOLO durante o processamento da base UTKFace.

São utilizados os arquivos CSV gerados nos notebooks de inferência, organizados em três partições para cada tecnologia.

A partir desses arquivos, são realizados os seguintes procedimentos:

- Carregamento dos resultados de cada tecnologia;
- Seleção dos registros com inferência bem-sucedida;
- Cálculo do Erro Absoluto Médio (MAE) por partição;
- Organização dos resultados em uma tabela comparativa;
- Exportação da tabela consolidada em formato CSV.

O cálculo do MAE utiliza a coluna "Erro Absoluto" já existente nos arquivos de resultados.


In [ ]:

from google.colab import drive

drive.mount("/content/drive")


In [ ]:

import os
import pandas as pd
import numpy as np

# ============================================================
# CAMINHOS DOS RESULTADOS
# ============================================================

PASTA_RESULTADOS = (
    "/content/drive/MyDrive/Classroom/"
    "9923 31 - TRABALHO DE CONCLUSÃO DE CURSO 2026/"
    "DESENVOLVIMENTO/Ambientes Colab/resultados"
)

PASTA_RESULTADOS_DEEPFACE = os.path.join(
    PASTA_RESULTADOS,
    "DeepFace",
    "csv"
)

PASTA_RESULTADOS_MIVOLO = os.path.join(
    PASTA_RESULTADOS,
    "MiVOLO",
    "arquivos csv"
)

PASTA_SAIDA = PASTA_RESULTADOS

# ============================================================
# ARQUIVOS OFICIAIS
# ============================================================

arquivos_resultados = {
    "DeepFace": {
        "Parte 1": os.path.join(
            PASTA_RESULTADOS_DEEPFACE,
            "resultados_deepface_utkface_part1.csv"
        ),
        "Parte 2": os.path.join(
            PASTA_RESULTADOS_DEEPFACE,
            "resultados_deepface_utkface_part2.csv"
        ),
        "Parte 3": os.path.join(
            PASTA_RESULTADOS_DEEPFACE,
            "resultados_deepface_utkface_part3.csv"
        )
    },

    "MiVOLO": {
        "Parte 1": os.path.join(
            PASTA_RESULTADOS_MIVOLO,
            "resultados_mivolo_utkface_part1.csv"
        ),
        "Parte 2": os.path.join(
            PASTA_RESULTADOS_MIVOLO,
            "resultados_mivolo_utkface_part2.csv"
        ),
        "Parte 3": os.path.join(
            PASTA_RESULTADOS_MIVOLO,
            "resultados_mivolo_utkface_part3.csv"
        )
    }
}

print("Configuração dos arquivos concluída.")


In [ ]:

# ============================================================
# CONSOLIDAÇÃO E CÁLCULO DO MAE
# ============================================================

resultados = []

for tecnologia, partes in arquivos_resultados.items():

    print(f"\n{'=' * 50}")
    print(f"TECNOLOGIA: {tecnologia}")
    print(f"{'=' * 50}")

    for parte, caminho in partes.items():

        print(f"\nProcessando: {parte}")

        # Verifica se o arquivo existe
        if not os.path.isfile(caminho):
            raise FileNotFoundError(
                f"Arquivo não encontrado: {caminho}"
            )

        # Carrega o CSV
        df = pd.read_csv(caminho)

        # Verifica as colunas necessárias
        colunas_necessarias = [
            "Erro Absoluto",
            "Status"
        ]

        for coluna in colunas_necessarias:
            if coluna not in df.columns:
                raise ValueError(
                    f"Coluna '{coluna}' não encontrada em {caminho}"
                )

        # Seleciona somente as inferências bem-sucedidas
        if tecnologia == "DeepFace":
            df_validos = df[
                df["Status"].astype(str).str.strip() == "Sucesso"
            ].copy()

        elif tecnologia == "MiVOLO":
            df_validos = df[
                df["Status"].astype(str).str.strip() == "OK"
            ].copy()

        # Converte os erros absolutos para valores numéricos
        df_validos["Erro Absoluto"] = pd.to_numeric(
            df_validos["Erro Absoluto"],
            errors="coerce"
        )

        # Remove somente valores ausentes ou não finitos
        df_validos = df_validos[
            np.isfinite(df_validos["Erro Absoluto"])
        ]

        # Verifica se existem resultados válidos
        if df_validos.empty:
            print("Nenhuma inferência válida encontrada.")
            continue

        # ====================================================
        # CÁLCULO DO ERRO ABSOLUTO MÉDIO (MAE)
        # ====================================================

        mae = df_validos["Erro Absoluto"].mean()

        resultados.append({
            "Tecnologia": tecnologia,
            "Base": "UTKFace",
            "Parte": parte,
            "Quantidade de inferências": len(df_validos),
            "MAE (anos)": round(mae, 2)
        })

        print(
            f"Inferências válidas: {len(df_validos)}"
        )

        print(
            f"MAE: {mae:.2f} anos"
        )

print("\nConsolidação concluída.")


In [ ]:

# ============================================================
# ORGANIZAÇÃO DA TABELA FINAL
# ============================================================

df_resultados = pd.DataFrame(resultados)

ordem_tecnologia = [
    "DeepFace",
    "MiVOLO"
]

ordem_parte = [
    "Parte 1",
    "Parte 2",
    "Parte 3"
]

df_resultados["Tecnologia"] = pd.Categorical(
    df_resultados["Tecnologia"],
    categories=ordem_tecnologia,
    ordered=True
)

df_resultados["Parte"] = pd.Categorical(
    df_resultados["Parte"],
    categories=ordem_parte,
    ordered=True
)

df_resultados = (
    df_resultados
    .sort_values(
        ["Tecnologia", "Parte"]
    )
    .reset_index(drop=True)
)

# Apresentação da tabela
print("=" * 65)
print("RESULTADOS CONSOLIDADOS — UTKFace")
print("=" * 65)

display(df_resultados)


In [ ]:

# ============================================================
# EXPORTAÇÃO DA TABELA CONSOLIDADA
# ============================================================

arquivo_saida = os.path.join(
    PASTA_SAIDA,
    "resultados_mae_utkface.csv"
)

df_resultados.to_csv(
    arquivo_saida,
    index=False,
    encoding="utf-8-sig"
)

print("Tabela consolidada salva com sucesso!")
print(f"\nLocal do arquivo:\n{arquivo_saida}")
